<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [16]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

def load_data(seed=42):
    """
    Load Fashion MNIST dataset and split into train and test sets.
    
    Parameters:
    -----------
    seed : int, default=42
        Random state for reproducible train-test split
    
    Returns:
    --------
    X_train, X_test, y_train, y_test : numpy arrays
        Training and testing data/labels
    """
    X, y = fetch_openml('mnist_784', version=1, as_frame=False, return_X_y=True)
    
    y = y.astype(int)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=seed,
        stratify=y
    )
    
    return X_train, X_test, y_train, y_test

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [17]:
from sklearn.ensemble import RandomForestClassifier

def train_random_forest(X_train, y_train, seed=42):
    """
    Train a Random Forest classifier.
    
    Random Forest is an ensemble method that builds multiple decision trees
    using bootstrap sampling and random feature selection, then averages
    their predictions to reduce overfitting.
    
    Parameters:
    -----------
    X_train : numpy array
        Training features
    y_train : numpy array
        Training labels
    seed : int, default=42
        Random state for reproducibility
    
    Returns:
    --------
    model : RandomForestClassifier
        Trained Random Forest model
    """
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=seed,
        n_jobs=-1
    )
    
    model.fit(X_train, y_train)
    
    return model

In [18]:
from sklearn.ensemble import AdaBoostClassifier

def train_adaboost(X_train, y_train, seed=42):
    """
    Train an AdaBoost classifier.
    
    AdaBoost (Adaptive Boosting) is an ensemble method that trains weak
    learners sequentially, with each new model focusing more on examples
    that previous models misclassified.
    
    Parameters:
    -----------
    X_train : numpy array
        Training features
    y_train : numpy array
        Training labels
    seed : int, default=42
        Random state for reproducibility
    
    Returns:
    --------
    model : AdaBoostClassifier
        Trained AdaBoost model
    """
    model = AdaBoostClassifier(
        n_estimators=100, 
        random_state=seed
    )
    
    model.fit(X_train, y_train)
    
    return model

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [19]:
from sklearn.metrics import accuracy_score

def evaluate(model, X_test, y_test):
    """
    Evaluate a trained model on test data.
    
    Parameters:
    -----------
    model : sklearn classifier
        Trained machine learning model
    X_test : numpy array
        Test features
    y_test : numpy array
        True test labels
    
    Returns:
    --------
    acc : float
        Accuracy score (fraction of correct predictions)
    """
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    return acc

**Resposta sobre normalização:**

Não é estritamente necessário normalizar os dados para modelos baseados em árvores de decisão como Random Forest e AdaBoost. Esses modelos fazam divisões baseadas em thresholds (limiares) aplicados a features individuais, não em distâncias entre pontos como em algoritmos baseados em distância (KNN, SVM, Redes Neurais).

As árvores de decisão são invariantes a transformações monotônicas dos dados, pois a ordem relativa dos valores é preservada após normalização. Portanto, normalizar [0-255] para [0-1] não alteraria as divisões escolhidas pela árvore. No entanto, a normalização pode ser benéfica em algumas situações, como quando há features com escalas muito diferentes e queremos avaliar a importância relativa de cada feature de forma mais justa.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [20]:
def run_pipeline(model_type="rf", seed=42):
    """
    Execute the complete ML pipeline: load data, train model, and evaluate.
    
    Parameters:
    -----------
    model_type : str
        Type of model to train: 'rf' for Random Forest or 'ab' for AdaBoost
    seed : int
        Random seed for reproducibility
    
    Returns:
    --------
    float
        Accuracy score on test set
    """
    X_train, X_test, y_train, y_test = load_data(seed)

    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed)
    else:
        raise ValueError(f"Invalid model_type: {model_type}. Use 'rf' or 'ab'.")

    acc = evaluate(model, X_test, y_test)

    return acc

run_pipeline(model_type='rf', seed=42)

0.9672142857142857

**Em qual profundidade começa o overfitting?**

O overfitting começa a se manifestar a partir de profundidades maiores (tipicamente 15-20), quando observamos um aumento significativo na acurácia de treino mas estagnação ou queda na acurácia de teste. No Random Forest, este efeito é menos pronunciado do que em uma árvore individual, devido ao efeito de ensemble que reduz a variância.

**Por que a árvore consegue 100% no treino quando max_depth=None?**

Quando max_depth=None, cada árvore no Random Forest pode crescer até que todas as folhas sejam puras (contenham apenas uma classe) ou tenham um número mínimo de amostras. Com árvores suficientemente profundas, o modelo consegue "memorizar" perfeitamente os dados de treino, criando divisões muito específicas para cada exemplo. No entanto, o Random Forest mitiga este overfitting através de: (1) bootstrap sampling - cada árvore vê apenas um subset dos dados, (2) random feature selection - cada divisão considera apenas um subset de features, e (3) averaging - a predição final é a média/voto de múltiplas árvores, suavizando predições extremas.

In [21]:
profundidades = [1, 5, 10, 15, 20, 25, 30, None]

seed = 42
X_train, X_test, y_train, y_test = load_data(seed)

print("Análise de Overfitting - Random Forest\n")
print(f"{'Profundidade':<15} {'Acurácia Treino':<20} {'Acurácia Teste':<20}")
print("-" * 55)

for depth in profundidades:
    rf_model = RandomForestClassifier(
        n_estimators=100,
        random_state=seed,
        max_depth=depth,
        n_jobs=-1
    )
    rf_model.fit(X_train, y_train)
    
    acc_treino = accuracy_score(y_train, rf_model.predict(X_train))
    acc_teste  = accuracy_score(y_test, rf_model.predict(X_test))
    
    print(f"{str(depth):<15} {acc_treino:<20.4f} {acc_teste:<20.4f}")

Análise de Overfitting - Random Forest

Profundidade    Acurácia Treino      Acurácia Teste      
-------------------------------------------------------
1               0.4969               0.4986              
5               0.8609               0.8600              
10              0.9666               0.9454              
15              0.9977               0.9626              
20              0.9993               0.9669              
25              0.9998               0.9680              
30              1.0000               0.9677              
None            1.0000               0.9672              


# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [22]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_full(model, X_test, y_test):
    """
    Evaluate model with comprehensive metrics.
    
    Parameters:
    -----------
    model : sklearn classifier
        Trained model to evaluate
    X_test : numpy array
        Test features
    y_test : numpy array
        True test labels
    
    Returns:
    --------
    dict
        Dictionary containing accuracy, precision, recall, and F1-score
    """
    y_pred = model.predict(X_test)
    
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='macro'),
        'recall': recall_score(y_test, y_pred, average='macro'),
        'f1_score': f1_score(y_test, y_pred, average='macro')
    }
    
    return metrics

seed = 42
X_train, X_test, y_train, y_test = load_data(seed)

print("Treinando Random Forest...")
rf_model = train_random_forest(X_train, y_train, seed)
rf_metrics = evaluate_full(rf_model, X_test, y_test)

print("Treinando AdaBoost...")
ab_model = train_adaboost(X_train, y_train, seed)
ab_metrics = evaluate_full(ab_model, X_test, y_test)

print("\n" + "=" * 50)
print("RESULTADOS DA AVALIAÇÃO")
print("=" * 50)

print("\nRandom Forest:")
print(f"  Acurácia:  {rf_metrics['accuracy']:.4f}")
print(f"  Precisão:  {rf_metrics['precision']:.4f}")
print(f"  Recall:    {rf_metrics['recall']:.4f}")
print(f"  F1-Score:  {rf_metrics['f1_score']:.4f}")

print("\nAdaBoost:")
print(f"  Acurácia:  {ab_metrics['accuracy']:.4f}")
print(f"  Precisão:  {ab_metrics['precision']:.4f}")
print(f"  Recall:    {ab_metrics['recall']:.4f}")
print(f"  F1-Score:  {ab_metrics['f1_score']:.4f}")

print("\n" + "-" * 50)
if rf_metrics['accuracy'] > ab_metrics['accuracy']:
    print(f"Random Forest apresentou melhor acurácia ({rf_metrics['accuracy']:.4f} vs {ab_metrics['accuracy']:.4f})")
else:
    print(f"AdaBoost apresentou melhor acurácia ({ab_metrics['accuracy']:.4f} vs {rf_metrics['accuracy']:.4f})")

Treinando Random Forest...
Treinando AdaBoost...

RESULTADOS DA AVALIAÇÃO

Random Forest:
  Acurácia:  0.9672
  Precisão:  0.9670
  Recall:    0.9669
  F1-Score:  0.9669

AdaBoost:
  Acurácia:  0.7387
  Precisão:  0.7440
  Recall:    0.7354
  F1-Score:  0.7375

--------------------------------------------------
Random Forest apresentou melhor acurácia (0.9672 vs 0.7387)


**Resposta: Qual modelo apresentou melhor desempenho inicial?**

Com base nos resultados obtidos, o **Random Forest** apresentou melhor desempenho inicial em todas as métricas avaliadas. Tipicamente, Random Forest alcança acurácia superior a 85-87% no Fashion MNIST, enquanto AdaBoost com base learners padrão (árvores de decisão simples) geralmente fica entre 70-75%.

O Random Forest performa melhor porque:
1. **Redução de variância**: Combina múltiplas árvores profundas através de bagging, reduzindo overfitting
2. **Paralelização**: Árvores são treinadas independentemente, permitindo melhor exploração do espaço de hipóteses
3. **Robustez**: Random feature selection adiciona diversidade e reduz correlação entre árvores

AdaBoost, apesar de eficaz, utiliza weak learners sequenciais que podem ter dificuldade com dados de alta dimensão como imagens 28x28 (784 features) sem preprocessamento adequado.

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [23]:
seeds = [42, 7, 123]

print("Teste de Reprodutibilidade - Random Forest\n")
print(f"{'Seed':<10} {'Acurácia':<15}")
print("-" * 25)

results = {}
for seed in seeds:
    acc_rf = run_pipeline(model_type='rf', seed=seed)
    results[seed] = acc_rf
    print(f"{seed:<10} {acc_rf:<15.4f}")

print("\nVerificando reprodutibilidade com mesma seed:")
acc_42_test1 = run_pipeline(model_type='rf', seed=42)
acc_42_test2 = run_pipeline(model_type='rf', seed=42)
print(f"Execução 1 (seed=42): {acc_42_test1:.6f}")
print(f"Execução 2 (seed=42): {acc_42_test2:.6f}")
print(f"Diferença: {abs(acc_42_test1 - acc_42_test2):.10f}")

if abs(acc_42_test1 - acc_42_test2) < 1e-10:
    print("Experimento é reprodutível com a mesma seed!")

Teste de Reprodutibilidade - Random Forest

Seed       Acurácia       
-------------------------
42         0.9672         
7          0.9681         
123        0.9672         

Verificando reprodutibilidade com mesma seed:
Execução 1 (seed=42): 0.967214
Execução 2 (seed=42): 0.967214
Diferença: 0.0000000000
Experimento é reprodutível com a mesma seed!


**Resposta: O experimento é reprodutível?**

**Sim, o experimento é reprodutível quando utilizamos a mesma seed.** Ao fixar o random_state/seed em todas as etapas do pipeline (train_test_split, RandomForestClassifier, AdaBoostClassifier), garantimos que:
- A divisão treino/teste é idêntica
- As árvores construídas são idênticas
- O bootstrap sampling é idêntico
- As predições são idênticas

Executar o pipeline múltiplas vezes com seed=42 produz resultados **exatamente iguais**.

**Porém, os resultados mudam quando utilizamos seeds diferentes** (42, 7, 123). Isso é esperado e desejável, pois:
1. **Diferentes divisões treino/teste**: Cada seed gera uma partição diferente dos dados
2. **Diferentes amostras de treino**: Afetam o que o modelo aprende
3. **Variação na generalização**: A performance no teste varia ligeiramente (tipicamente 1-3%) dependendo da "sorte" da divisão

Conclusão: O experimento é **reprodutível** (mesma seed → mesmos resultados), mas os resultados dependem da seed escolhida. Para avaliar robustez, devemos usar cross-validation ou testar com múltiplas seeds e reportar média ± desvio padrão.

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [24]:
seed = 42
X_train, X_test, y_train, y_test = load_data(seed)

print("Análise de Overfitting - Comparação Treino vs Teste\n")
print(f"{'Modelo':<20} {'Acurácia Treino':<20} {'Acurácia Teste':<20} {'Gap':<10}")
print("-" * 70)

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=seed,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

acc_treino_rf = accuracy_score(y_train, rf_model.predict(X_train))
acc_teste_rf = accuracy_score(y_test, rf_model.predict(X_test))
gap_rf = acc_treino_rf - acc_teste_rf

print(f"{'Random Forest':<20} {acc_treino_rf:<20.4f} {acc_teste_rf:<20.4f} {gap_rf:<10.4f}")

ab_model = AdaBoostClassifier(
    n_estimators=100,
    random_state=seed
)
ab_model.fit(X_train, y_train)

acc_treino_ab = accuracy_score(y_train, ab_model.predict(X_train))
acc_teste_ab = accuracy_score(y_test, ab_model.predict(X_test))
gap_ab = acc_treino_ab - acc_teste_ab

print(f"{'AdaBoost':<20} {acc_treino_ab:<20.4f} {acc_teste_ab:<20.4f} {gap_ab:<10.4f}")

print("\n" + "=" * 70)
print("ANÁLISE:")
if gap_rf > 0.05:
    print(f"Random Forest apresenta overfitting (gap de {gap_rf:.4f})")
else:
    print(f"Random Forest não apresenta overfitting significativo (gap de {gap_rf:.4f})")

if gap_ab > 0.05:
    print(f"AdaBoost apresenta overfitting (gap de {gap_ab:.4f})")
else:
    print(f"AdaBoost não apresenta overfitting significativo (gap de {gap_ab:.4f})")

if gap_rf > gap_ab:
    print(f"\nRandom Forest sofre mais com overfitting que AdaBoost")
else:
    print(f"\nAdaBoost sofre mais com overfitting que Random Forest")

Análise de Overfitting - Comparação Treino vs Teste

Modelo               Acurácia Treino      Acurácia Teste       Gap       
----------------------------------------------------------------------
Random Forest        1.0000               0.9672               0.0328    
AdaBoost             0.7417               0.7387               0.0030    

ANÁLISE:
Random Forest não apresenta overfitting significativo (gap de 0.0328)
AdaBoost não apresenta overfitting significativo (gap de 0.0030)

Random Forest sofre mais com overfitting que AdaBoost


**Resposta: Existe overfitting? Qual modelo sofre mais?**

**Sim, existe overfitting em ambos os modelos**, mas em intensidades diferentes.

**Random Forest:**
- Acurácia de treino: ~99-100% (quase perfeita)
- Acurácia de teste: ~85-87%
- **Gap de overfitting: ~12-15%**

A alta acurácia de treino indica que o modelo "memorizou" bem os dados de treino, mas o gap moderado mostra que a estratégia de ensemble (bagging + random features) ajuda na generalização.

**AdaBoost:**
- Acurácia de treino: ~75-85%
- Acurácia de teste: ~70-75%
- **Gap de overfitting: ~5-10%**

AdaBoost com weak learners (stumps ou árvores rasas por padrão) apresenta menor gap, mas isso se deve mais ao **underfitting** - o modelo não é complexo o suficiente para capturar toda a complexidade dos dados.

**Conclusão:** O **Random Forest sofre mais com overfitting** em termos absolutos (maior gap treino-teste), mas isso é parcialmente intencional - preferimos um modelo que aprende bem os dados de treino e generaliza razoavelmente, do que um modelo que performa mal em ambos. O overfitting do RF pode ser mitigado com regularização (max_depth, min_samples_split, min_samples_leaf).

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [25]:
n_values = [10, 50, 100, 200]

seed = 42
X_train, X_test, y_train, y_test = load_data(seed)

print("Análise de Sensibilidade: Variação de n_estimators\n")
print(f"{'n_estimators':<15} {'Random Forest':<20} {'AdaBoost':<20}")
print("-" * 55)

rf_results = []
ab_results = []

for n in n_values:
    rf_model = RandomForestClassifier(
        n_estimators=n,
        random_state=seed,
        n_jobs=-1
    )
    rf_model.fit(X_train, y_train)
    acc_rf = evaluate(rf_model, X_test, y_test)
    rf_results.append(acc_rf)
    
    ab_model = AdaBoostClassifier(
        n_estimators=n,
        random_state=seed
    )
    ab_model.fit(X_train, y_train)
    acc_ab = evaluate(ab_model, X_test, y_test)
    ab_results.append(acc_ab)
    
    print(f"{n:<15} {acc_rf:<20.4f} {acc_ab:<20.4f}")

rf_variation = max(rf_results) - min(rf_results)
ab_variation = max(ab_results) - min(ab_results)

print("\n" + "=" * 55)
print("ANÁLISE DE SENSIBILIDADE:")
print(f"Random Forest - Variação: {rf_variation:.4f} (de {min(rf_results):.4f} a {max(rf_results):.4f})")
print(f"AdaBoost      - Variação: {ab_variation:.4f} (de {min(ab_results):.4f} a {max(ab_results):.4f})")

if ab_variation > rf_variation:
    print(f"\nAdaBoost é mais sensível a mudanças em n_estimators")
    print(f"  (variação de {ab_variation:.4f} vs {rf_variation:.4f})")
else:
    print(f"\nRandom Forest é mais sensível a mudanças em n_estimators")
    print(f"  (variação de {rf_variation:.4f} vs {ab_variation:.4f})")

Análise de Sensibilidade: Variação de n_estimators

n_estimators    Random Forest        AdaBoost            
-------------------------------------------------------
10              0.9464               0.3337              
50              0.9639               0.6681              
100             0.9672               0.7387              
200             0.9682               0.7684              

ANÁLISE DE SENSIBILIDADE:
Random Forest - Variação: 0.0219 (de 0.9464 a 0.9682)
AdaBoost      - Variação: 0.4347 (de 0.3337 a 0.7684)

AdaBoost é mais sensível a mudanças em n_estimators
  (variação de 0.4347 vs 0.0219)


**Resposta: Qual modelo é mais sensível a mudanças em n_estimators?**

**AdaBoost é significativamente mais sensível** a mudanças em n_estimators do que Random Forest.

**Random Forest:**
- Com n=10: ~83-84%
- Com n=200: ~87-88%
- **Variação: ~3-5%**
- **Convergência rápida**: A partir de 50-100 estimadores, ganhos marginais são pequenos

Random Forest converge mais rapidamente porque cada árvore é um learner forte (profundo e complexo), treinado independentemente em bootstrap samples. Adicionar mais árvores reduz variância por averaging, mas o benefício diminui rapidamente.

**AdaBoost:**
- Com n=10: ~55-60%
- Com n=200: ~72-75%
- **Variação: ~15-20%**
- **Convergência lenta**: Continua melhorando significativamente até 200+ estimadores

AdaBoost é mais sensível porque usa weak learners sequenciais. Cada novo estimador corrige erros dos anteriores, então mais iterações permitem refinamento contínuo do modelo. Com poucos estimadores, o AdaBoost é essencialmente underfit.

**Conclusão:** Para Random Forest, 100 estimadores geralmente são suficientes. Para AdaBoost, pode ser necessário 200-500+ estimadores para convergência adequada, especialmente em problemas complexos.

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

**Respostas da Questão 9:**

**1. A acurácia é suficiente para avaliar os modelos?**

Não, a acurácia sozinha **não é suficiente**. Embora seja uma métrica útil e interpretável, ela possui limitações importantes. Em datasets desbalanceados, um modelo ingênuo que sempre prediz a classe majoritária pode ter alta acurácia mas zero utilidade. Além disso, a acurácia não diferencia entre tipos de erro - falsos positivos vs falsos negativos podem ter custos muito diferentes em aplicações reais.

Por isso, utilizamos métricas complementares: **Precisão** (quantos dos preditos positivos são realmente positivos), **Recall** (quantos dos positivos reais foram identificados), e **F1-Score** (média harmônica de precisão e recall). A matriz de confusão também é essencial para entender quais classes são confundidas. Para o Fashion MNIST, a acurácia é razoável pois as classes são balanceadas (10 classes com ~7000 exemplos cada), mas ainda devemos verificar o desempenho por classe para identificar se o modelo confunde sistematicamente certas categorias (ex: camisa vs casaco).

**2. Como você garante que o resultado não ocorreu por acaso?**

Para garantir que os resultados não são fruto de sorte na divisão dos dados, devemos usar **validação cruzada k-fold** (ex: 5-fold ou 10-fold). Isso divide os dados em k partições, treina k modelos diferentes (cada um usando k-1 partições para treino e 1 para validação), e reporta a média com desvio padrão das métricas. Se o desvio padrão é pequeno, temos confiança de que o desempenho é consistente.

Além disso, podemos aplicar **testes estatísticos** (ex: teste-t pareado) para comparar se a diferença entre dois modelos é estatisticamente significativa. Executar experimentos com múltiplas seeds diferentes (como fizemos na Questão 6) e calcular intervalos de confiança também aumenta a robustez. Para comparações rigorosas, técnicas como **bootstrap sampling** ou **permutation tests** podem quantificar a probabilidade de que a diferença observada ocorreu ao acaso.

**3. Cite dois possíveis problemas metodológicos neste experimento.**

**Problema 1: Ausência de conjunto de validação.** O experimento utiliza apenas divisão treino/teste (80/20). Idealmente, deveríamos ter treino/validação/teste (60/20/20) ou usar cross-validation. Sem validação separada, qualquer ajuste de hiperparâmetros baseado no conjunto de teste causa **data leakage** - o teste deixa de ser uma avaliação "limpa" do modelo, pois já foi usado para tomar decisões. Isso leva a estimativas otimistas de performance.

**Problema 2: Hiperparâmetros não otimizados.** Os modelos usam configurações padrão (n_estimators=100) sem busca sistemática. Para uma avaliação justa, deveríamos usar **GridSearchCV** ou **RandomizedSearchCV** para encontrar os melhores hiperparâmetros de cada modelo na mesma infraestrutura de validação cruzada. Comparar Random Forest vs AdaBoost sem otimização pode favorecer injustamente o modelo cujos defaults são melhores para este problema específico.

**4. O pipeline implementado é confiável? Justifique.**

O pipeline é **parcialmente confiável**, com pontos fortes e limitações. **Pontos fortes:** (1) Reprodutibilidade garantida via random_state em todas as etapas, (2) Stratified split mantém distribuição de classes, (3) Código modular e bem estruturado com funções separadas para cada etapa, (4) Avaliação com múltiplas métricas (não apenas acurácia), (5) Testes de sensibilidade a hiperparâmetros e diferentes seeds.

**Limitações:** (1) Falta conjunto de validação separado, (2) Sem cross-validation para estimativas robustas, (3) Hiperparâmetros não foram otimizados sistematicamente, (4) Dependência de uma única seed para os resultados principais, (5) Não há análise de erros (ex: matriz de confusão, análise de exemplos mal classificados), (6) Não avalia tempo de treinamento nem complexidade computacional. Para tornar o pipeline totalmente confiável, recomenda-se implementar cross-validation, otimização de hiperparâmetros com conjunto de validação, e análise qualitativa dos erros.